# T4 · Build DimCity
Subtasks: Block 1 (conforming map), Block 2 (DimCity with the -1 row), self-checks, three judgement questions. All in this one notebook.

## Block 1 · Conforming map (23 raw strings)

In [1]:
import pandas as pd, os
pd.set_option("display.max_colwidth", None, "display.width", 250, "display.max_rows", 100)

# raw string -> (CityName, FacilityType). Port city? = the CITY has a seaport (a dry port is not a port city).
raw_map = [
 ("Karachi Port",           "Karachi",    "Seaport"),
 ("Port Qasim",             "Karachi",    "Seaport"),
 ("Gwadar Port",            "Gwadar",     "Seaport"),
 ("Karachi Cantt",          "Karachi",    "Rail Terminal"),
 ("Pipri Marshalling Yard", "Karachi",    "Rail Yard"),
 ("Lahore",                 "Lahore",     "City"),
 ("Karachi",                "Karachi",    "City"),
 ("Multan",                 "Multan",     "City"),
 ("Islamabad",              "Islamabad",  "City"),
 ("Quetta",                 "Quetta",     "City"),
 ("Faisalabad",             "Faisalabad", "City"),
 ("Rawalpindi",             "Rawalpindi", "City"),
 ("Lahore Dry Port",        "Lahore",     "Dry Port"),
 ("Faisalabad Dry Port",    "Faisalabad", "Dry Port"),
 ("Sialkot Dry Port",       "Sialkot",    "Dry Port"),
 ("Peshawar",               "Peshawar",   "City"),
 ("Islamabad Dry Port",     "Islamabad",  "Dry Port"),
 ("Hyderabad",              "Hyderabad",  "City"),
 ("Peshawar Cantt",         "Peshawar",   "Rail Terminal"),
 ("Sukkur",                 "Sukkur",     "City"),
 ("Sialkot",                "Sialkot",    "City"),
 ("Gujranwala",             "Gujranwala", "City"),
 ("Abbottabad",             "Abbottabad", "City"),
]

# city attributes, stored once per city: (Province, Region, Port city?)
# Region = PakFreight corridor grouping (editorial choice): South / West / Central / North
city_attr = {
 "Karachi":    ("Sindh",                       "South",   "Y"),
 "Hyderabad":  ("Sindh",                       "South",   "N"),
 "Sukkur":     ("Sindh",                       "South",   "N"),
 "Gwadar":     ("Balochistan",                 "West",    "Y"),
 "Quetta":     ("Balochistan",                 "West",    "N"),
 "Lahore":     ("Punjab",                      "Central", "N"),
 "Faisalabad": ("Punjab",                      "Central", "N"),
 "Multan":     ("Punjab",                      "Central", "N"),
 "Sialkot":    ("Punjab",                      "Central", "N"),
 "Gujranwala": ("Punjab",                      "Central", "N"),
 "Rawalpindi": ("Punjab",                      "Central", "N"),
 "Islamabad":  ("Islamabad Capital Territory", "North",   "N"),
 "Peshawar":   ("Khyber Pakhtunkhwa",          "North",   "N"),
 "Abbottabad": ("Khyber Pakhtunkhwa",          "North",   "N"),
}

block1 = pd.DataFrame(raw_map, columns=["Raw location string", "Conformed CityName", "FacilityType"])
block1.insert(0, "#", range(1, 24))
block1["Province"]   = block1["Conformed CityName"].map(lambda c: city_attr[c][0])
block1["Region"]     = block1["Conformed CityName"].map(lambda c: city_attr[c][1])
block1["Port city?"] = block1["Conformed CityName"].map(lambda c: city_attr[c][2])
block1 = block1[["#", "Raw location string", "Conformed CityName", "Province", "Region", "Port city?", "FacilityType"]]
block1

,#,Raw location string,Conformed CityName,Province,Region,Port city?,FacilityType
0,1,Karachi Port,Karachi,Sindh,South,Y,Seaport
1,2,Port Qasim,Karachi,Sindh,South,Y,Seaport
2,3,Gwadar Port,Gwadar,Balochistan,West,Y,Seaport
3,4,Karachi Cantt,Karachi,Sindh,South,Y,Rail Terminal
4,5,Pipri Marshalling Yard,Karachi,Sindh,South,Y,Rail Yard
5,6,Lahore,Lahore,Punjab,Central,N,City
6,7,Karachi,Karachi,Sindh,South,Y,City
7,8,Multan,Multan,Punjab,Central,N,City
8,9,Islamabad,Islamabad,Islamabad Capital Territory,North,N,City
9,10,Quetta,Quetta,Balochistan,West,N,City


## Block 2 · DimCity (one row per city + reserved -1 row)

In [2]:
tier = {"Seaport": 0, "Dry Port": 1, "Rail Terminal": 2, "Rail Yard": 3, "City": 4}   # city-level FacilityType = highest tier present

cities = list(dict.fromkeys(block1["Conformed CityName"]))          # first-appearance order -> Karachi = 1
rows = []
for key, city in enumerate(cities, start=1):
    g = block1[block1["Conformed CityName"] == city]
    prov, reg, port = city_attr[city]
    ftype = min(g.FacilityType, key=tier.get)
    rows.append((key, city, prov, reg, port, ftype, "; ".join(g["Raw location string"])))
rows.append((-1, "Unknown", "Unknown", "Unknown", "N", "Unknown", ""))

DimCity = pd.DataFrame(rows, columns=["CityKey", "CityName", "Province", "Region", "Port?", "FacilityType", "SourceAliases"])
DimCity

,CityKey,CityName,Province,Region,Port?,FacilityType,SourceAliases
0,1,Karachi,Sindh,South,Y,Seaport,Karachi Port; Port Qasim; Karachi Cantt; Pipri Marshalling Yard; Karachi
1,2,Gwadar,Balochistan,West,Y,Seaport,Gwadar Port
2,3,Lahore,Punjab,Central,N,Dry Port,Lahore; Lahore Dry Port
3,4,Multan,Punjab,Central,N,City,Multan
4,5,Islamabad,Islamabad Capital Territory,North,N,Dry Port,Islamabad; Islamabad Dry Port
5,6,Quetta,Balochistan,West,N,City,Quetta
6,7,Faisalabad,Punjab,Central,N,Dry Port,Faisalabad; Faisalabad Dry Port
7,8,Rawalpindi,Punjab,Central,N,City,Rawalpindi
8,9,Sialkot,Punjab,Central,N,Dry Port,Sialkot Dry Port; Sialkot
9,10,Peshawar,Khyber Pakhtunkhwa,North,N,Rail Terminal,Peshawar; Peshawar Cantt


## Self-checks (all three must pass before T5)

In [3]:
real = DimCity[DimCity.CityKey != -1]
sc = pd.DataFrame([
 ("Distinct conformed city names in Block 1",        14, block1["Conformed CityName"].nunique()),
 ("Raw strings that map to Karachi",                  5, int((block1["Conformed CityName"] == "Karachi").sum())),
 ("Rows in DimCity, excluding the Unknown row",      14, len(real)),
], columns=["Check", "Expected", "Your result"])
sc["Pass?"] = (sc.Expected == sc["Your result"]).map({True: "pass", False: "FIX"})
assert (sc["Pass?"] == "pass").all()
assert DimCity.loc[DimCity.CityName == "Karachi", "CityKey"].item() == 1          # Karachi must be key 1 everywhere
assert DimCity.CityKey.is_unique and (DimCity.CityKey == -1).sum() == 1
sc

,Check,Expected,Your result,Pass?
0,Distinct conformed city names in Block 1,14,14,pass
1,Raw strings that map to Karachi,5,5,pass
2,"Rows in DimCity, excluding the Unknown row",14,14,pass


## Verification against the data (T7 checks that depend on DimCity)

In [4]:
p = next(x for x in ["Shipments_Unified.csv", "../Shipments_Unified.csv", "/mnt/user-data/uploads/Shipments_Unified.csv"] if os.path.exists(x))
src = pd.read_csv(p)

alias_key = {a: r.CityKey for r in DimCity.itertuples() for a in r.SourceAliases.split("; ") if a}
lookup = lambda s: alias_key.get(s, -1)               # unmatched string -> Unknown (-1), never dropped

f = src.copy()
f["OriginCityKey"] = f.Origin.map(lookup)
f["DestCityKey"]   = f.Destination.map(lookup)
f = f.merge(DimCity[["CityKey", "Province"]].rename(columns={"CityKey": "OriginCityKey", "Province": "OriginProvince"}), on="OriginCityKey", how="left")

unknown = int((f.OriginCityKey == -1).sum() + (f.DestCityKey == -1).sum())
t7 = pd.DataFrame([
 ("#12 Fact rows pointing at Unknown (-1)",  0,       unknown),
 ("#9  WeightKG, Sindh / Marine (origin)",   169_000, int(f[(f.OriginProvince == "Sindh") & (f.SourceSystem == "Marine")].WeightKG.sum())),
 ("#10 WeightKG, Punjab / Road (origin)",    44_100,  int(f[(f.OriginProvince == "Punjab") & (f.SourceSystem == "Road")].WeightKG.sum())),
 ("#11 WeightKG, Sindh, all modes (origin)", 484_350, int(f[f.OriginProvince == "Sindh"].WeightKG.sum())),
 ("#2  Grand total WeightKG (rows kept)",    578_650, int(f.WeightKG.sum())),
], columns=["T7 check", "Expected", "Result"])
t7["Pass?"] = (t7.Expected == t7.Result).map({True: "pass", False: "FIX"})
assert (t7["Pass?"] == "pass").all() and len(f) == 34
print("New spelling 'KHI Port Trust' ->", lookup("KHI Port Trust"))
t7

New spelling 'KHI Port Trust' -> -1


,T7 check,Expected,Result,Pass?
0,#12 Fact rows pointing at Unknown (-1),0,0,pass
1,"#9 WeightKG, Sindh / Marine (origin)",169000,169000,pass
2,"#10 WeightKG, Punjab / Road (origin)",44100,44100,pass
3,"#11 WeightKG, Sindh, all modes (origin)",484350,484350,pass
4,#2 Grand total WeightKG (rows kept),578650,578650,pass


## Judgement questions

**Q1. Pipri Marshalling Yard merged into Karachi: one business decision this quietly made.**

In [5]:
rail = src[src.SourceSystem == "Rail"].copy()
rail["Yard"] = rail.Origin
rail["Delayed"] = (rail.Status == "Delayed").astype(int)
rail.groupby("Yard").agg(legs=("ShipmentID", "count"), delayed=("Delayed", "sum"), kg=("WeightKG", "sum"))

,legs,delayed,kg
Yard,,,
Karachi Cantt,6,0,178000
Pipri Marshalling Yard,4,2,117250


PakFreight decided it will not judge its two Karachi rail facilities separately. In the data, both delayed rail legs (RL-45016, RL-45018) left Pipri (2 of 4 legs delayed) while Karachi Cantt has 0 of 6 delayed, yet any DimCity-based report shows one "Karachi" rail origin with 2 delayed of 10. A yard-level problem, such as who to hold accountable or where to add capacity, becomes invisible. If that matters later, the fix is a facility-level level below City (a hierarchy), not another conforming guess.

**Q2. Islamabad is its own territory, not part of Punjab.**

In [6]:
wrong_prov = f.assign(P=f.OriginProvince.replace({"Islamabad Capital Territory": "Punjab"}))
print("Punjab / Road if Islamabad is mapped to Punjab:", int(wrong_prov[(wrong_prov.P == "Punjab") & (wrong_prov.SourceSystem == "Road")].WeightKG.sum()), "(correct: 44,100)")
blank = f.assign(P=f.OriginProvince.replace({"Islamabad Capital Territory": None}))
print("Origin weight with Province blank, shown as (Blank):", int(blank[blank.P.isna()].WeightKG.sum()), "kg")

Punjab / Road if Islamabad is mapped to Punjab: 50200 (correct: 44,100)
Origin weight with Province blank, shown as (Blank): 6100 kg


**Mapped to Punjab:** Punjab is overstated and the T7 Punjab/Road check fails (50,200 kg instead of 44,100, because RD-505 leaves Islamabad), and every provincial or federal-vs-provincial report is wrong without any error showing. **Left blank:** the weight does not vanish, but it lands in an unlabelled "(Blank)" row that nobody can tell from missing data; province slicers exclude it, province-level reports stop adding up to the national total on screen, and "Islamabad ICT" can't appear in the T7 matrix. The right answer is a real value, `Islamabad Capital Territory`, in the province column.

**Q3. A new file arrives with "KHI Port Trust".** In order:
1. **Load dimensions first, then facts.** The staging lookup against `SourceAliases` finds no match, so the fact row gets `OriginCityKey = -1` instead of being dropped (shown above: `lookup("KHI Port Trust")` returns -1). The raw string itself is never edited.
2. **The -1 count surfaces it.** T7 check #12 reads non-zero; the Conformer lists the unmapped string.
3. **Decide what it is.** It is the Karachi seaport, an alias of an existing city, so no new row is minted and the key stays `CityKey = 1`. Only a genuinely new city would get the next surrogate key (15) with province, region, port flag and facility type filled in.
4. **Add the alias** to Karachi's `SourceAliases` (the mapping).
5. **Re-run the fact load** (or re-key the rows currently at -1) so those rows pick up key 1.
6. **Re-run T7:** row count and totals unchanged, -1 count back to 0, `DimCity` still 14 rows (excluding Unknown).